# 01 Bronze Ingest -- Vision Zero LA

**Workspace:** CTS-Geo Analytics GBD  
**Lakehouse:** VisionZero_Lakehouse  
**Folder:** Vision_Zero  

**Purpose:** Fetch raw LAPD traffic collision data and LA school locations from
the City of Los Angeles SODA API and write them into Bronze-layer Delta tables
in the Lakehouse. No cleaning or transformation is performed here -- that is
handled by `02_silver_clean_enrich`.

**Data Sources:**
- LAPD Traffic Collision Data: https://data.lacity.org/resource/d5tf-ez2w.json
- LA School Locations: https://data.lacity.org/resource/qh44-jvjp.json

**Outputs:**
- `bronze_crashes` Delta table (partitioned by crash_year)
- `bronze_schools` Delta table

In [ ]:
%pip install h3 requests

In [ ]:
import requests
import json
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType
)

# ── Configuration ─────────────────────────────────────────────────────────────
SODA_CRASH_URL = "https://data.lacity.org/resource/d5tf-ez2w.json"
SODA_SCHOOL_URL = "https://data.lacity.org/resource/qh44-jvjp.json"
DATE_FROM = "2022-01-01"
BATCH_SIZE = 50000

LAKEHOUSE = "VisionZero_Lakehouse"

print(f"Bronze ingest configured.")
print(f"  Crash API : {SODA_CRASH_URL}")
print(f"  School API: {SODA_SCHOOL_URL}")
print(f"  Date range: {DATE_FROM} to present")
print(f"  Lakehouse : {LAKEHOUSE}")

In [ ]:
# ── Fetch LAPD crash data with pagination ─────────────────────────────────────

def fetch_lapd_crashes(date_from, batch_size=50000):
    """Fetch all LAPD crash records from SODA API with offset-based pagination."""
    all_records = []
    offset = 0

    while True:
        url = (
            f"{SODA_CRASH_URL}"
            f"?$where=date_occ>'{date_from}T00:00:00'"
            f"&$limit={batch_size}"
            f"&$offset={offset}"
            f"&$select=dr_no,date_occ,time_occ,area_name,crm_cd_desc,"
            f"vict_age,vict_sex,vict_descent,premis_desc,location,"
            f"cross_street,lat,lon,mocodes"
            f"&$order=date_occ DESC"
        )

        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f"  ERROR fetching offset {offset}: {e}")
            break

        batch = resp.json()
        if not batch:
            break

        all_records.extend(batch)
        batch_num = offset // batch_size + 1
        print(f"  Batch {batch_num}: fetched {len(batch)} records "
              f"(cumulative {len(all_records)})")

        if len(batch) < batch_size:
            break
        offset += batch_size

    return all_records


print(f"Fetching LAPD crashes from {DATE_FROM}...")
raw_records = fetch_lapd_crashes(DATE_FROM, BATCH_SIZE)
print(f"Total records fetched: {len(raw_records)}")

In [ ]:
# ── Create Spark DataFrame with explicit schema ────────────────────────────────

crash_schema = StructType([
    StructField("dr_no", StringType()),
    StructField("date_occ", StringType()),
    StructField("time_occ", StringType()),
    StructField("area_name", StringType()),
    StructField("crm_cd_desc", StringType()),
    StructField("vict_age", StringType()),
    StructField("vict_sex", StringType()),
    StructField("vict_descent", StringType()),
    StructField("premis_desc", StringType()),
    StructField("location", StringType()),
    StructField("cross_street", StringType()),
    StructField("lat", StringType()),
    StructField("lon", StringType()),
    StructField("mocodes", StringType()),
])

crashes_raw = spark.createDataFrame(raw_records, crash_schema)

row_count = crashes_raw.count()
print(f"Raw Spark DataFrame created: {row_count} rows")
crashes_raw.printSchema()

In [ ]:
# ── Write to bronze_crashes Delta table (partitioned by crash_year) ────────────

crashes_bronze = (
    crashes_raw
    .withColumn("crash_year",
                F.year(F.to_date(F.col("date_occ"))))
)

(
    crashes_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("crash_year")
    .saveAsTable(f"{LAKEHOUSE}.bronze_crashes")
)

print(f"Saved {row_count} rows to {LAKEHOUSE}.bronze_crashes")

In [ ]:
# ── Fetch and ingest LA school locations ──────────────────────────────────────

def fetch_school_locations(batch_size=50000):
    """Fetch LA school location records from SODA API."""
    all_records = []
    offset = 0

    while True:
        url = (
            f"{SODA_SCHOOL_URL}"
            f"?$limit={batch_size}"
            f"&$offset={offset}"
        )

        try:
            resp = requests.get(url, timeout=120)
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f"  ERROR fetching schools at offset {offset}: {e}")
            break

        batch = resp.json()
        if not batch:
            break

        all_records.extend(batch)
        print(f"  Fetched {len(all_records)} school records")

        if len(batch) < batch_size:
            break
        offset += batch_size

    return all_records


print("Fetching LA school locations...")
school_records = fetch_school_locations()
print(f"Total school records: {len(school_records)}")

if school_records:
    schools_df = spark.createDataFrame(school_records)
    (
        schools_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{LAKEHOUSE}.bronze_schools")
    )
    school_count = schools_df.count()
    print(f"Saved {school_count} rows to {LAKEHOUSE}.bronze_schools")
else:
    school_count = 0
    print("WARNING: No school records fetched. Check the SODA endpoint.")

In [ ]:
# ── Summary ─────────────────────────────────────────────────────────────────────

crash_count = spark.read.format("delta").table(f"{LAKEHOUSE}.bronze_crashes").count()

try:
    school_ct = spark.read.format("delta").table(f"{LAKEHOUSE}.bronze_schools").count()
except Exception:
    school_ct = 0

print("=" * 60)
print("Bronze Ingest Summary")
print("=" * 60)
print(f"  bronze_crashes : {crash_count:,} rows")
print(f"  bronze_schools : {school_ct:,} rows")
print("=" * 60)
print("Bronze layer complete. Proceed to 02_silver_clean_enrich.")